# Silver Layer - Repository Metadata

This notebook transforms the Bronze Repository Metadata table into the Silver layer.

### Objectives
- Extract required attributes from the raw GitHub JSON.
- Convert date fields into Timestamp format.
- Rename columns for consistency.
- Add processing metadata.
- Remove duplicate repositories.
- Load the cleaned data into the Silver table.

In [0]:
from pyspark.sql import functions as F

bronze_repo = spark.table("gitobservatory.bronze.repository_metadata")

silver_repo = bronze_repo.select(
    F.col("github_repo_id").alias("repo_id"),
    "owner",
    "repository_name",
    "repo_full_name",
    F.concat_ws("/",F.col("owner"),F.col("repository_name")).alias("repository_key"),
    "source_system",
    F.col("ingestion_timestamp").alias("bronze_loaded_timestamp"),
    F.get_json_object("raw_json", "$.description").alias("description"),
    F.get_json_object("raw_json", "$.language").alias("language"),
    F.get_json_object("raw_json", "$.visibility").alias("visibility"),
    F.get_json_object("raw_json", "$.default_branch").alias("default_branch"),
    F.get_json_object("raw_json", "$.homepage").alias("homepage"),
    F.get_json_object("raw_json", "$.license.name").alias("license_name"),
    F.to_timestamp(F.get_json_object("raw_json", "$.created_at")).alias("created_at"),
    F.to_timestamp(F.get_json_object("raw_json", "$.updated_at")).alias("updated_at"),
    F.to_timestamp(F.get_json_object("raw_json", "$.pushed_at")).alias("pushed_at"),
    F.get_json_object("raw_json","$.size").cast("int").alias("size"),
    F.get_json_object("raw_json","$.stargazers_count").cast("int").alias("stars"),
    F.get_json_object("raw_json","$.forks_count").cast("int").alias("forks"),
    F.get_json_object("raw_json","$.watchers_count").cast("int").alias("watchers"),
    F.get_json_object("raw_json","$.subscribers_count").cast("int").alias("subscribers"),
    F.get_json_object("raw_json","$.open_issues_count").cast("int").alias("open_issues"),
    F.get_json_object("raw_json","$.has_issues").cast("boolean").alias("has_issues"),
    F.get_json_object("raw_json","$.has_projects").cast("boolean").alias("has_projects"),
    F.get_json_object("raw_json","$.has_wiki").cast("boolean").alias("has_wiki"),
    F.get_json_object("raw_json","$.has_discussions").cast("boolean").alias("has_discussions"),
    F.get_json_object("raw_json","$.fork").cast("boolean").alias("is_fork"),
    F.get_json_object("raw_json","$.archived").cast("boolean").alias("is_archived"),
    F.get_json_object("raw_json","$.disabled").cast("boolean").alias("is_disabled")
)

silver_repo = silver_repo.dropDuplicates(["repo_id"])
silver_repo = silver_repo.withColumn("silver_processed_timestamp",F.current_timestamp())

silver_repo.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.silver.repositories")

In [0]:
from pyspark.sql import functions as F

bronze_pr = spark.table("gitobservatory.bronze.pull_requests")

silver_pr = bronze_pr.select(
    "pr_id",
    "pr_node_id",
    "pr_number",
    "repo_id",
    "repo_node_id",
    "repo_full_name",
    "user_id",
    "user_login",
    "author_association",
    "state",
    "draft",
    "html_url",
    "source_system",
    F.col("ingestion_timestamp").alias("bronze_loaded_timestamp"),
    F.get_json_object("raw_json", "$.title").alias("title"),
    F.get_json_object("raw_json", "$.body").alias("body"),
    F.get_json_object("raw_json", "$.comments").cast("int").alias("comments"),
    F.get_json_object("raw_json", "$.review_comments").cast("int").alias("review_comments"),
    F.get_json_object("raw_json", "$.commits").cast("int").alias("commits"),
    F.get_json_object("raw_json", "$.additions").cast("int").alias("additions"),
    F.get_json_object("raw_json", "$.deletions").cast("int").alias("deletions"),
    F.get_json_object("raw_json", "$.changed_files").cast("int").alias("changed_files"),
    F.get_json_object("raw_json", "$.mergeable").cast("boolean").alias("mergeable"),
    F.get_json_object("raw_json", "$.mergeable_state").alias("mergeable_state"),
    F.when(
    F.col("merged_at").isNotNull(),
    True
).otherwise(False).alias("merged"),
    F.get_json_object("raw_json", "$.locked").cast("boolean").alias("locked"),
    F.get_json_object("raw_json", "$.maintainer_can_modify").cast("boolean").alias("maintainer_can_modify"),
    F.get_json_object("raw_json", "$.labels").alias("labels"),
    F.get_json_object("raw_json", "$.assignees").alias("assignees"),
    F.get_json_object("raw_json", "$.requested_reviewers").alias("requested_reviewers"),
    F.get_json_object("raw_json", "$.milestone").alias("milestone"),
    F.to_timestamp("created_at").alias("created_at"),
    F.to_timestamp("updated_at").alias("updated_at"),
    F.to_timestamp("closed_at").alias("closed_at"),
    F.to_timestamp("merged_at").alias("merged_at")
)
silver_pr = silver_pr.withColumn(
    "merge_time_hours",
    F.when(
        F.col("merged"),
        (F.unix_timestamp("merged_at") -
         F.unix_timestamp("created_at")) / 3600
    )
)
silver_pr = silver_pr.dropDuplicates(["pr_id"])
silver_pr = silver_pr.withColumn("silver_processed_timestamp",F.current_timestamp())
silver_pr.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.silver.pull_requests")

# Silver Layer - Issues

This notebook transforms the Bronze Issues table into the Silver layer.

**Objectives**
- Extract issue attributes from the raw GitHub JSON.
- Convert date fields into Timestamp format.
- Calculate issue resolution time and issue age.
- Generate analytical flags for milestones, labels, assignees and issue status.
- Remove duplicate issues.
- Add processing metadata.
- Load the cleaned data into the Silver layer.

In [0]:
from pyspark.sql import functions as F

bronze_issue = spark.table("gitobservatory.bronze.issues")

silver_issue = bronze_issue.select(
    "issue_id",
    "issue_node_id",
    "issue_number",
    "repo_id",
    "repo_node_id",
    "repo_full_name",
    "user_id",
    "user_login",
    "author_association",
    "state",
    "state_reason",
    "comments",
    "html_url",
    "source_system",
    F.col("ingestion_timestamp").alias("bronze_loaded_timestamp"),
    F.get_json_object("raw_json", "$.title").alias("title"),
    F.get_json_object("raw_json", "$.body").alias("body"),
    F.get_json_object("raw_json", "$.labels").alias("labels"),
    F.get_json_object("raw_json", "$.assignees").alias("assignees"),
    F.get_json_object("raw_json", "$.milestone").alias("milestone"),
    F.get_json_object("raw_json", "$.locked").cast("boolean").alias("locked"),
    F.get_json_object("raw_json", "$.active_lock_reason").alias("active_lock_reason"),
    F.get_json_object("raw_json","$.reactions.total_count").cast("int").alias("reaction_count"),
    F.to_timestamp("created_at").alias("created_at"),
    F.to_timestamp("updated_at").alias("updated_at"),
    F.to_timestamp("closed_at").alias("closed_at")
)
silver_issue = silver_issue.withColumn("issue_resolution_time_hours",(F.unix_timestamp("closed_at")- F.unix_timestamp("created_at")) / 3600)
silver_issue = silver_issue.withColumn("issue_age_days",F.datediff(F.current_timestamp(), F.col("created_at")))
silver_issue = silver_issue.withColumn("is_closed",F.col("state") == "closed")
silver_issue = silver_issue.withColumn("has_milestone",F.col("milestone").isNotNull())

silver_issue = silver_issue.withColumn("has_assignees",(F.col("assignees").isNotNull()) &(F.col("assignees") != "[]"))

silver_issue = silver_issue.withColumn("has_labels",(F.col("labels").isNotNull()) &(F.col("labels") != "[]"))

silver_issue = silver_issue.dropDuplicates(["issue_id"])

silver_issue = silver_issue.withColumn("silver_processed_timestamp",F.current_timestamp())

silver_issue.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.silver.issues")

# Silver Layer - Contributors

This notebook transforms the Bronze Contributors table into the Silver layer.

**Objectives**
- Extract contributor profile information from the raw GitHub JSON.
- Standardize contributor attributes.
- Classify contributors based on contribution activity.
- Generate analytical flags for contributor profiling.
- Remove duplicate contributor records.
- Add processing metadata.
- Load the cleaned data into the Silver layer.

In [0]:
from pyspark.sql import functions as F

bronze_contributor = spark.table("gitobservatory.bronze.contributors")
silver_contributor = bronze_contributor.select(
    "contributor_id",
    "contributor_node_id",
    "repo_id",
    "repo_node_id",
    "repo_full_name",
    "user_login",
    "user_type",
    "site_admin",
    "contributions",
    "html_url",
    "source_system",
    F.col("ingestion_timestamp").alias("bronze_loaded_timestamp"),
    F.get_json_object("raw_json", "$.avatar_url").alias("avatar_url"),
    F.get_json_object("raw_json", "$.organizations_url").alias("organizations_url"),
    F.get_json_object("raw_json", "$.repos_url").alias("repos_url"),
    F.get_json_object("raw_json", "$.followers_url").alias("followers_url"),
    F.get_json_object("raw_json", "$.following_url").alias("following_url"),
    F.get_json_object("raw_json", "$.received_events_url").alias("received_events_url")
)
silver_contributor = silver_contributor.withColumn("is_bot",F.col("user_type") == "Bot")

silver_contributor = silver_contributor.withColumn("is_site_admin",F.col("site_admin"))

silver_contributor = silver_contributor.withColumn("profile_available",F.col("html_url").isNotNull())

silver_contributor = silver_contributor.withColumn("is_top_contributor",F.col("contributions") >= 100)

silver_contributor = silver_contributor.withColumn("contribution_level",
    F.when(F.col("contributions") >= 500, "Expert")
     .when(F.col("contributions") >= 100, "Advanced")
     .when(F.col("contributions") >= 10, "Intermediate")
     .otherwise("Beginner")
)

silver_contributor = silver_contributor.withColumn(
    "contribution_score",
    F.when(F.col("contributions") >= 500, 4)
     .when(F.col("contributions") >= 100, 3)
     .when(F.col("contributions") >= 10, 2)
     .otherwise(1)
)

silver_contributor = silver_contributor.withColumn("is_external",~F.col("is_bot"))

silver_contributor = silver_contributor.dropDuplicates(["contributor_id", "repo_id"])

silver_contributor = silver_contributor.withColumn("silver_processed_timestamp",F.current_timestamp())

silver_contributor.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.silver.contributors")



# Silver Layer - Pull Request Reviews

This notebook transforms the Bronze Pull Request Reviews table into the Silver layer.

**Objectives**
- Convert review timestamps into Timestamp format.
- Remove duplicate review records.
- Preserve review information for analytics.
- Add processing metadata.
- Load the cleaned data into the Silver layer.

In [0]:
from pyspark.sql import functions as F

bronze_review = spark.table("gitobservatory.bronze.pull_request_reviews")

silver_review = bronze_review.select(
    "review_id",
    "pr_id",
    "reviewer_id",
    "reviewer_login",
    "state",
    F.to_timestamp("submitted_at").alias("submitted_at"),
    "body",
    "html_url",
    "source_system",
    F.col("ingestion_timestamp").alias("bronze_loaded_timestamp")
)

silver_review = silver_review.dropDuplicates(["review_id"])
silver_review = silver_review.withColumn("is_approved",F.col("state") == "APPROVED")
silver_review = silver_review.withColumn("changes_requested",F.col("state") == "CHANGES_REQUESTED")
silver_review = silver_review.withColumn("is_comment",F.col("state") == "COMMENTED")
silver_review = silver_review.withColumn("silver_processed_timestamp",F.current_timestamp())
silver_review.write.format("delta").option("overwriteSchema","true").mode("overwrite").saveAsTable("gitobservatory.silver.pull_request_reviews")

# Silver Layer - Workflow Runs

This notebook transforms the Bronze Workflow Runs table into the Silver layer.

**Objectives**
- Convert workflow timestamps into Timestamp format.
- Remove duplicate workflow runs.
- Generate workflow status indicators.
- Add processing metadata.
- Load the cleaned data into the Silver layer.

In [0]:
from pyspark.sql import functions as F

bronze_workflow = spark.table("gitobservatory.bronze.workflow_runs")
silver_workflow = bronze_workflow.select(
    "run_id",
    "workflow_id",
    "repo_id",
    "repo_full_name",
    "name",
    "status",
    "conclusion",
    F.to_timestamp("created_at").alias("created_at"),
    F.to_timestamp("updated_at").alias("updated_at"),
    "run_number",
    "source_system",
    F.col("ingestion_timestamp").alias("bronze_loaded_timestamp")
)

silver_workflow = silver_workflow.dropDuplicates(["run_id"])
silver_workflow = silver_workflow.withColumn("workflow_duration_minutes",(F.unix_timestamp("updated_at") -F.unix_timestamp("created_at")) / 60)
silver_workflow = silver_workflow.withColumn("is_success",F.col("conclusion") == "success")
silver_workflow = silver_workflow.withColumn("is_failure",F.col("conclusion") == "failure")
silver_workflow = silver_workflow.withColumn("is_cancelled",F.col("conclusion") == "cancelled")
silver_workflow = silver_workflow.withColumn("silver_processed_timestamp",F.current_timestamp())

silver_workflow.write.format("delta").option("overwriteSchema","true").mode("overwrite").saveAsTable("gitobservatory.silver.workflow_runs")
